In [ ]:
#1 Notebook description:

# this notebook is used to evaluate the market of assets and find potential assets to invest in
# with the best risk/reward characteristics. This notebook is intented to analyze a broad group of market assets
# and does NOT focus on any particular assets. Computing data for a large number of assets is computationally expensive and thus
# the analysis is relegated to other notebooks
# this notebook now also replaces the old 'Market Study - Statistical' notebook through the preset options below

In [ ]:
#2 Load libraries
import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
# Load libraries
import sys
import os
project_path = os.getcwd()
while not os.path.exists(os.path.join(project_path, "pyproject.toml")):
    parent = os.path.dirname(project_path)
    if parent == project_path:
        raise FileNotFoundError("Could not locate the project root from the current working directory.")
    project_path = parent
if project_path not in sys.path:
    sys.path.append(project_path)
from Quantapp.visualization import Plotter
from Quantapp.analytics import Rolling, RiskRelativeAnalytics, SignalLabels, SeriesTransforms
from Quantapp.data import MacroDataClient
from Quantapp.data import MarketDataClient

import numpy as np
import json
import pandas as pd
import yfinance as yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.subplots as sp
import plotly.graph_objects as go
import pandas as pd
import holidays
import plotly.express as px
import concurrent.futures
from plotly.subplots import make_subplots
import plotly.graph_objects as go

#shut down warnings
import warnings
warnings.filterwarnings("ignore")
pio.templates.default = 'plotly_dark'
plt.style.use('dark_background')



rolling = Rolling()
risk_relative_analytics = RiskRelativeAnalytics()
signal_labels = SignalLabels()
series_transforms = SeriesTransforms()
qp = Plotter()
qe = MacroDataClient()
market_data = MarketDataClient()
                

In [ ]:
#3 Parameters
time_frame_week = 7
time_frame_short = 21
time_frame_mid   = 50
time_frame_long = 200
interval = '1d'
period     = '10y'
risk_free_rate = 0.02 / 252  # Annualized risk-free rate divided by trading days
benchmark = 'SPY'
mode='standard'
analysis_preset = 'broad'  # supported: 'broad', 'statistical'
preset_config = {
    'broad': {'asset_class': 'broad', 'sector': 'all'},
    'statistical': {'asset_class': 'equity', 'sector': 'financials'},
}
asset_class = preset_config[analysis_preset]['asset_class']
sector = preset_config[analysis_preset]['sector']
# Optional manual overrides:
# asset_class = 'equity'
# sector = 'healthcare'

In [ ]:
#4 Broad market reference: normalized prices, rolling Sharpe, rolling Sharpe z-scores, and Sharpe spread z-scores
broad_market_reference = market_data.get_broad_market_data()

broad_market_close = pd.DataFrame({
    name: df['Close']
    for name, df in broad_market_reference.items()
    if not df.empty and 'Close' in df
}).sort_index()

def _normalize_series(series):
    valid = series.dropna()
    if valid.empty:
        return series
    return series.div(valid.iloc[0]).mul(100)

def plot_reference_dataframe(data, title, yaxis_title, add_zero_line=False, sigma_lines=None):
    if data.empty:
        print(f'No data available for {title}.')
        return

    plot_data = data.dropna(how='all')
    end_date = plot_data.index.max()
    min_date = plot_data.index.min()
    zoom_years = [1, 2, 3, 5, 10]
    default_years = 2
    default_start = max(min_date, end_date - pd.DateOffset(years=default_years))
    zoom_buttons = []
    for years in zoom_years:
        start_date = max(min_date, end_date - pd.DateOffset(years=years))
        zoom_buttons.append(
            dict(
                label=f'{years}y',
                method='relayout',
                args=[{'xaxis.range': [start_date, end_date]}]
            )
        )

    fig = go.Figure()
    for column in plot_data.columns:
        fig.add_trace(
            go.Scatter(
                x=plot_data.index,
                y=plot_data[column],
                mode='lines',
                name=column
            )
        )

    if add_zero_line:
        fig.add_hline(y=0, line_dash='dash', line_color='gray')
    if sigma_lines:
        for sigma in sigma_lines:
            fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)')
            fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)')

    fig.update_layout(
        title=title,
        xaxis_title='Date',
        yaxis_title=yaxis_title,
        height=500,
        template='plotly_dark',
        autosize=True,
        margin=dict(l=40, r=40, t=90, b=40),
        updatemenus=[
            dict(
                buttons=zoom_buttons,
                direction='down',
                showactive=True,
                active=1,
                x=0.01,
                y=1.18,
                xanchor='left',
                yanchor='top'
            )
        ]
    )
    fig.update_xaxes(range=[default_start, end_date])
    fig.show(config={'responsive': True})

if broad_market_close.empty:
    print('No broad market reference data available.')
else:
    broad_market_close = broad_market_close.ffill()
    broad_market_normalized = broad_market_close.apply(_normalize_series).dropna(how='all')

    broad_market_daily_returns = broad_market_close.pct_change()
    broad_market_excess_returns = broad_market_daily_returns.sub(risk_free_rate)
    broad_market_rolling_sharpe = broad_market_excess_returns.rolling(time_frame_long).mean().div(
        broad_market_daily_returns.rolling(time_frame_long).std()
    ).mul(np.sqrt(252)).dropna(how='all')

    broad_market_rolling_sharpe_zscore = broad_market_rolling_sharpe.sub(
        broad_market_rolling_sharpe.rolling(time_frame_long).mean()
    ).div(
        broad_market_rolling_sharpe.rolling(time_frame_long).std()
    ).dropna(how='all')

    broad_market_reference_benchmark = 'U.S. Large Cap'
    if broad_market_reference_benchmark not in broad_market_rolling_sharpe.columns:
        broad_market_reference_benchmark = broad_market_rolling_sharpe.columns[0]

    broad_market_sharpe_spread = broad_market_rolling_sharpe.sub(
        broad_market_rolling_sharpe[broad_market_reference_benchmark],
        axis=0
    )
    broad_market_sharpe_spread_zscore = broad_market_sharpe_spread.sub(
        broad_market_sharpe_spread.rolling(time_frame_long).mean()
    ).div(
        broad_market_sharpe_spread.rolling(time_frame_long).std()
    ).dropna(how='all')
    broad_market_sharpe_spread_zscore = broad_market_sharpe_spread_zscore.drop(
        columns=[broad_market_reference_benchmark],
        errors='ignore'
    )

    plot_reference_dataframe(
        broad_market_normalized,
        'Broad Market Reference - Normalized Prices',
        'Normalized Price (Start = 100)'
    )
    plot_reference_dataframe(
        broad_market_rolling_sharpe,
        'Broad Market Reference - Rolling Sharpe (200 Day)',
        'Rolling Sharpe',
        add_zero_line=True
    )
    plot_reference_dataframe(
        broad_market_rolling_sharpe_zscore,
        'Broad Market Reference - Rolling Sharpe Z-Scores (200 Day)',
        'Rolling Sharpe Z-Score',
        add_zero_line=True,
        sigma_lines=[1, 2, 3]
    )
    plot_reference_dataframe(
        broad_market_sharpe_spread_zscore,
        f'Broad Market Reference - Sharpe Spread Z-Scores vs {broad_market_reference_benchmark}',
        'Sharpe Spread Z-Score',
        add_zero_line=True
    )


In [ ]:
#5 Load: retrieve all tickers / prices 
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Load: retrieve all tickers / prices / spreads for major markets
sp500 = yf.download('SPY', period=period, interval=interval,progress=False)
risk_free_rate = yf.download('^IRX', period=period, interval=interval, progress=False)
#market_assets = qe.get_market_assets()

market_assets = market_data.get_market_assets()

if asset_class == 'broad':
    #Load: retrieve prices
    indices_df             = market_data.generate_series(market_assets['INDICES'], columns=['Close'],period=period, interval=interval)
    sectors_df            = market_data.generate_series(market_assets['SECTORS'], columns=['Close'],period=period, interval=interval)
    industries_df         = market_data.generate_series(market_assets['INDUSTRIES'], columns=['Close'],period=period, interval=interval)
    bonds_df = market_data.generate_series(market_assets['BONDS'], columns=['Close'],period=period, interval=interval)
    precious_metals_df = market_data.generate_series(market_assets['PRECIOUS_METALS'], columns=['Close'],period=period, interval=interval)
    crypto_df = market_data.generate_series(market_assets['CRYPTO'], columns=['Close'],period=period, interval=interval)
    #crypto_df = crypto_df.loc[sp500.index]
    energy_df = market_data.generate_series(market_assets['ENERGY'], columns=['Close'],period=period, interval=interval)
    foreign_markets_df = market_data.generate_series(market_assets['FOREIGN_MARKETS'], columns=['Close'],period=period, interval=interval)
    primary_sector_etfs_df = market_data.generate_series(market_assets['PRIMARY_SECTORS'], columns=['Close'],period=period, interval=interval)
    capitalizations_df = market_data.generate_series(market_assets['CAPITALIZATIONS'], columns=['Close'],period=period, interval=interval)
    innovation_df = market_data.generate_series(market_assets['INNOVATION'], columns=['Close'],period=period, interval=interval)
    long_leveraged_df = market_data.generate_series(market_assets['LONG_LEVERAGE'], columns=['Close'],period=period, interval=interval)
    short_leveraged_df = market_data.generate_series(market_assets['SHORT_LEVERAGE'], columns=['Close'],period=period, interval=interval)
    single_factor_df = market_data.generate_series(market_assets['SINGLE_FACTOR'], columns=['Close'],period=period, interval=interval)
    multi_factor_df = market_data.generate_series(market_assets['MULTI_FACTOR'], columns=['Close'],period=period, interval=interval)
    minimum_volatility_df = market_data.generate_series(market_assets['MINIMUM_VOLATILITY'], columns=['Close'],period=period, interval=interval)


    etf_prices = pd.concat([
        indices_df,
        sectors_df,
        industries_df,
        bonds_df,
        precious_metals_df,
        #crypto_df,
        energy_df,
        foreign_markets_df,
        primary_sector_etfs_df,
        capitalizations_df,
        innovation_df,
        long_leveraged_df,
        short_leveraged_df,
        single_factor_df,
        multi_factor_df,
        minimum_volatility_df
    ], axis=1).loc[:, lambda df: ~df.columns.duplicated()]

    #test
    test_prices = pd.concat([indices_df,
                            sectors_df,
                            crypto_df
                            ], axis=1).loc[:, lambda df: ~df.columns.duplicated()]

    benchmark_series           = etf_prices[benchmark]

    # List of dataframes for week and short time frames
    etf_dataframes = {
        "indices": indices_df, 
        "sectors": sectors_df, 
        "industries": industries_df, 
        "bonds": bonds_df, 
        "precious_metals": precious_metals_df, 
        #"crypto": crypto_df, 
        "energy": energy_df,
        "foreign_markets": foreign_markets_df, 
        "primary_sector_etfs": primary_sector_etfs_df, 
        "capitalizations": capitalizations_df, 
        "innovation": innovation_df,
        "long_leveraged": long_leveraged_df,
        "short_leveraged": short_leveraged_df,
        "single_factor": single_factor_df,
        "multi_factor": multi_factor_df,
        "minimum_volatility": minimum_volatility_df
    }
 
    print('Computing the correlation matrix...')
    etf_dataframes_correlation_matrices = {key: value.corr() for key, value in etf_dataframes.items()}
elif asset_class == 'equity':
    xlk_holdings_df = market_data.generate_series(market_assets['XLK_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlf_holdings_df = market_data.generate_series(market_assets['XLF_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xli_holdings_df = market_data.generate_series(market_assets['XLI_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlb_holdings_df = market_data.generate_series(market_assets['XLB_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlv_holdings_df = market_data.generate_series(market_assets['XLV_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlu_holdings_df = market_data.generate_series(market_assets['XLU_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xly_holdings_df = market_data.generate_series(market_assets['XLY_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlc_holdings_df = market_data.generate_series(market_assets['XLC_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlp_holdings_df = market_data.generate_series(market_assets['XLP_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xle_holdings_df = market_data.generate_series(market_assets['XLE_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlre_holdings_df = market_data.generate_series(market_assets['XLRE_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    indices_df = market_data.generate_series(market_assets['INDICES'], columns=['Close'], period=period, interval=interval)
    etf_prices = pd.concat([indices_df], axis=1).loc[:, lambda df: ~df.columns.duplicated()]
    benchmark_series = etf_prices[benchmark]
    etf_dataframes = {
        "indices": indices_df,
        "xlk_holdings": xlk_holdings_df,
        "xlf_holdings": xlf_holdings_df,
        "xli_holdings": xli_holdings_df,
        "xlb_holdings": xlb_holdings_df,
        "xlv_holdings": xlv_holdings_df,
        "xlu_holdings": xlu_holdings_df,
        "xly_holdings": xly_holdings_df,
        "xlc_holdings": xlc_holdings_df,
        "xlp_holdings": xlp_holdings_df,
        "xle_holdings": xle_holdings_df,
        "xlre_holdings": xlre_holdings_df
    }
    # Mapping of sectors to their respective ETF holdings dataframes
    sector_mapping = {
        'energy': ('xle_holdings', xle_holdings_df),
        'materials': ('xlb_holdings', xlb_holdings_df),
        'industrials': ('xli_holdings', xli_holdings_df),
        'consumer_discretionary': ('xly_holdings', xly_holdings_df),
        'consumer_staples': ('xlp_holdings', xlp_holdings_df),
        'healthcare': ('xlv_holdings', xlv_holdings_df),
        'financials': ('xlf_holdings', xlf_holdings_df),
        'information_technology': ('xlk_holdings', xlk_holdings_df),
        'communication_services': ('xlc_holdings', xlc_holdings_df),
        'utilities': ('xlu_holdings', xlu_holdings_df),
        'real_estate': ('xlre_holdings', xlre_holdings_df)
    }
    
    if sector == 'all':
        # Include all sectors by updating the etf_dataframes dictionary
        etf_dataframes.update({key: df for key, df in sector_mapping.values()})
    elif sector in sector_mapping:
        # Include only the specific sector
        key, df = sector_mapping[sector]
        etf_dataframes = {
            "indices": indices_df,
            key: df
        }
        etf_prices = pd.concat([indices_df, df], axis=1).loc[:, lambda df: ~df.columns.duplicated()]
    else:
        supported_values = "'all', " + ", ".join([f"'{s}'" for s in sector_mapping.keys()])
        raise ValueError(f"Unsupported sector. Supported values are {supported_values}.")
    etf_dataframes_correlation_matrices = {key: value.corr() for key, value in etf_dataframes.items()}    
    
else:
    raise ValueError("Unsupported asset class. Supported values are 'broad' and 'equity'.")


agg_dataframe = pd.concat(list(etf_dataframes.values()), axis=1)
agg_dataframe = agg_dataframe.loc[:,~agg_dataframe.columns.duplicated()]

In [ ]:
#6 Computations...

#1. return spreads between benchmark and all assets
#2. the sortino ratios for all assets
#3. the spreads between the sortino ratios of all assets and the benchmark


#compute the weekly, short, mid, and long term returns for the benchmark
sp500_monthly_returns = series_transforms.returns(sp500,frequency='monthly')
sp500_weekly_returns  = series_transforms.returns(sp500,frequency='weekly')
sp500_daily_returns   = series_transforms.returns(sp500,frequency='daily')

#the spreads between the benchmark and all assets
#-----------------------------------------------------------
#the spreads between the benchmark and all assets
#Create spreads for weekly time frame
print("Computing spreads between benchmark and all assets...")
benchmark_minus_etf_week = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=time_frame_week, mode=mode)

#display(benchmark_minus_etf_week)
#Create spreads for short time frame
print(f"Computing spreads between benchmark and all assets for the short time frame ({time_frame_short} days)...")
benchmark_minus_etf_short = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=time_frame_short, mode=mode)

#Create spreads for mid time frame
print(f"Computing spreads between benchmark and all assets for the mid time frame ({time_frame_mid} days)...")
benchmark_minus_etf_mid = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=time_frame_mid, mode=mode)

#create spreads for long time frame
print(f"Computing spreads between benchmark and all assets for the long time frame ({time_frame_long} days)...")
benchmark_minus_etf_long = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=time_frame_long, mode=mode)

print(" ")
print("-----------------------------------------------------------------------")

#-----------------------------------------------------------

#the sortino ratios for all assets
print(f"computing the rolling sortino ratios for all assets for the short time frame ({time_frame_short} days)...")
rolling_sortino_ratios_etf_21 = rolling.risk_adjusted_returns(etf_prices, windows=[21], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(f"computing the rolling sortino ratios for all assets for the mid time frame ({time_frame_mid} days)...")
rolling_sortino_ratios_etf_50 = rolling.risk_adjusted_returns(etf_prices, windows=[50], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(f"computing the rolling sortino ratios for all assets for the long time frame ({time_frame_long} days)...")
rolling_sortino_ratios_etf_200 = rolling.risk_adjusted_returns(etf_prices, windows=[200], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(" ")
print("-----------------------------------------------------------------------")

#the spreads between the sortino ratios of all assets and the benchmark
print(f"computing the rolling sortino ratios for all assets minus the benchmark for the short time frame ({time_frame_short} days)...")
rolling_sortino_ratios_benchmark_minus_etf_21  = -rolling_sortino_ratios_etf_21.sub(rolling_sortino_ratios_etf_21['SPY'], axis=0)

print(f"computing the rolling sortino ratios for all assets minus the benchmark for the mid time frame ({time_frame_mid} days)...")
rolling_sortino_ratios_benchmark_minus_etf_50  = -rolling_sortino_ratios_etf_50.sub(rolling_sortino_ratios_etf_50['SPY'], axis=0)

print(f"computing the rolling sortino ratios for all assets minus the benchmark for the long time frame ({time_frame_long} days)...")
rolling_sortino_ratios_benchmark_minus_etf_200  = -rolling_sortino_ratios_etf_200.sub(rolling_sortino_ratios_etf_200['SPY'], axis=0)

print(f"computing 10 year correlation metrics to {benchmark}...")
etf_daily_returns = etf_prices.pct_change(fill_method=None)
correlation_end = etf_daily_returns.index.max()
correlation_start = correlation_end - pd.DateOffset(years=10)
correlation_window = etf_daily_returns.loc[correlation_start:correlation_end]
correlation_tolerance = pd.Timedelta(days=45)
minimum_downside_observations = 30
correlation_metrics_10y = {
    'Pearson': {},
    'Spearman': {},
    'Kendall': {},
    'Downside Pearson': {}
}

for ticker in correlation_window.columns:
    aligned_returns = pd.concat([
        correlation_window[benchmark],
        correlation_window[ticker]
    ], axis=1).dropna()

    has_full_lookback = (
        not aligned_returns.empty
        and aligned_returns.index.min() <= correlation_start + correlation_tolerance
    )

    if has_full_lookback:
        benchmark_returns = aligned_returns.iloc[:, 0]
        asset_returns = aligned_returns.iloc[:, 1]
        downside_mask = benchmark_returns < 0
        downside_returns = aligned_returns.loc[downside_mask]

        correlation_metrics_10y['Pearson'][ticker] = benchmark_returns.corr(asset_returns, method='pearson')
        correlation_metrics_10y['Spearman'][ticker] = benchmark_returns.corr(asset_returns, method='spearman')
        correlation_metrics_10y['Kendall'][ticker] = benchmark_returns.corr(asset_returns, method='kendall')
        correlation_metrics_10y['Downside Pearson'][ticker] = (
            downside_returns.iloc[:, 0].corr(downside_returns.iloc[:, 1], method='pearson')
            if len(downside_returns) >= minimum_downside_observations else np.nan
        )
    else:
        for metric_name in correlation_metrics_10y:
            correlation_metrics_10y[metric_name][ticker] = np.nan

correlation_metrics_10y = {
    metric_name: pd.Series(metric_values)
    for metric_name, metric_values in correlation_metrics_10y.items()
}
correlation_10y = correlation_metrics_10y['Pearson']
'''
print(" ")
print("-----------------------------------------------------------------------")

print("Computing pairwise spreads between assets within each category...")
print(f"Computing pairwise spreads for the short time frame ({time_frame_short} days)...")
pairwise_spreads_21 = rolling.create_pairwise_spreads(etf_dataframes, window=time_frame_short)

print(f"Computing pairwise spreads for the mid time frame ({time_frame_mid} days)...")
pairwise_spreads_50 = rolling.create_pairwise_spreads(etf_dataframes, window=time_frame_mid)

print(f"Computing pairwise spreads for the long time frame ({time_frame_long} days)...")
pairwise_spreads_200 = rolling.create_pairwise_spreads(etf_dataframes, window=time_frame_long)

'''

In [ ]:
#7 Correlation Comparison by Metric
correlation_palette = {
    'Q1 (Lowest)': '#ef4444',
    'Q2': '#f59e0b',
    'Q3': '#60a5fa',
    'Q4 (Highest)': '#22c55e'
}
correlation_plot_frames = {}
pearson_correlation_series = correlation_metrics_10y.get('Pearson', pd.Series(dtype=float))

for metric_name, metric_series in correlation_metrics_10y.items():
    metric_column = f'10 year {metric_name} Correlation to {benchmark}'
    metric_plot_df = metric_series.rename(metric_column).dropna().rename_axis('Ticker').reset_index()
    if metric_plot_df.empty:
        continue

    pearson_rank_reference = pearson_correlation_series.reindex(metric_plot_df['Ticker']).dropna().sort_values()
    pearson_rank_map = pd.Series(
        np.arange(1, len(pearson_rank_reference) + 1),
        index=pearson_rank_reference.index
    )

    quantile_count = min(4, len(metric_plot_df))
    quantile_labels = ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'][:quantile_count]
    metric_plot_df['Quantile'] = pd.qcut(
        metric_plot_df[metric_column].rank(method='first'),
        q=quantile_count,
        labels=quantile_labels
    )
    metric_plot_df = metric_plot_df.sort_values(by=metric_column, ascending=True).reset_index(drop=True)
    metric_plot_df['Position'] = np.arange(len(metric_plot_df))
    metric_plot_df['Current Rank'] = np.arange(1, len(metric_plot_df) + 1)
    metric_plot_df['Pearson Rank'] = metric_plot_df['Ticker'].map(pearson_rank_map)
    metric_plot_df['Rank Shift vs Pearson'] = metric_plot_df['Pearson Rank'] - metric_plot_df['Current Rank']
    metric_plot_df['Metric'] = metric_name
    metric_plot_df['Current Rank Label'] = metric_plot_df['Current Rank'].astype(int).astype(str)
    metric_plot_df['Pearson Rank Label'] = metric_plot_df['Pearson Rank'].apply(lambda value: 'N/A' if pd.isna(value) else str(int(value)))
    metric_plot_df['Rank Shift Label'] = metric_plot_df['Rank Shift vs Pearson'].apply(
        lambda value: 'No change'
        if pd.isna(value) or value == 0
        else f'Up {int(abs(value))} positions' if value > 0
        else f'Down {int(abs(value))} positions'
    )
    metric_plot_df['Color'] = metric_plot_df['Quantile'].map(correlation_palette)
    metric_plot_df['Shift Color'] = metric_plot_df['Rank Shift vs Pearson'].apply(
        lambda value: '#22c55e' if value > 0 else '#ef4444' if value < 0 else '#9ca3af'
    )
    correlation_plot_frames[metric_name] = (metric_column, metric_plot_df)

if correlation_plot_frames:
    default_metric = 'Pearson' if 'Pearson' in correlation_plot_frames else next(iter(correlation_plot_frames))
    default_column, default_plot_df = correlation_plot_frames[default_metric]
    default_positions = default_plot_df['Position'].tolist()
    default_ticker_order = default_plot_df['Ticker'].tolist()

    correlation_fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=('Selected Correlation Metric', 'Rank Shift vs Pearson Order')
    )
    correlation_fig.add_trace(
        go.Bar(
            x=default_plot_df['Position'],
            y=default_plot_df[default_column],
            marker_color=default_plot_df['Color'],
            customdata=np.array(default_plot_df[['Ticker', 'Metric', 'Quantile', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']]),
            hovertemplate='Ticker: %{customdata[0]}<br>Metric: %{customdata[1]}<br>Value: %{y:.2f}<br>Quantile: %{customdata[2]}<br>Current Rank: %{customdata[3]}<br>Pearson Rank: %{customdata[4]}<br>Shift vs Pearson: %{customdata[5]}<extra></extra>'
        ),
        row=1,
        col=1
    )
    correlation_fig.add_trace(
        go.Bar(
            x=default_plot_df['Position'],
            y=default_plot_df['Rank Shift vs Pearson'],
            marker_color=default_plot_df['Shift Color'],
            customdata=np.array(default_plot_df[['Ticker', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']]),
            hovertemplate='Ticker: %{customdata[0]}<br>Rank Shift: %{y:+.0f}<br>Current Rank: %{customdata[1]}<br>Pearson Rank: %{customdata[2]}<br>Interpretation: %{customdata[3]}<extra></extra>'
        ),
        row=2,
        col=1
    )
    correlation_fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)

    metric_buttons = []
    for metric_name, (metric_column, metric_plot_df) in correlation_plot_frames.items():
        metric_buttons.append(
            dict(
                label=metric_name,
                method='update',
                args=[
                    {
                        'x': [metric_plot_df['Position'], metric_plot_df['Position']],
                        'y': [metric_plot_df[metric_column], metric_plot_df['Rank Shift vs Pearson']],
                        'marker.color': [metric_plot_df['Color'], metric_plot_df['Shift Color']],
                        'customdata': [
                            np.array(metric_plot_df[['Ticker', 'Metric', 'Quantile', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']]),
                            np.array(metric_plot_df[['Ticker', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']])
                        ]
                    },
                    {
                        'title': f'10 Year {metric_name} Correlation to {benchmark} by Quantile',
                        'xaxis.tickmode': 'array',
                        'xaxis.tickvals': metric_plot_df['Position'].tolist(),
                        'xaxis.ticktext': metric_plot_df['Ticker'].tolist(),
                        'xaxis2.tickmode': 'array',
                        'xaxis2.tickvals': metric_plot_df['Position'].tolist(),
                        'xaxis2.ticktext': metric_plot_df['Ticker'].tolist(),
                        'yaxis.title.text': 'Correlation',
                        'yaxis2.title.text': 'Rank Shift'
                    }
                ]
            )
        )

    correlation_fig.update_layout(
        title=f'10 Year {default_metric} Correlation to {benchmark} by Quantile',
        template='plotly_dark',
        autosize=True,
        height=760,
        margin=dict(l=40, r=40, t=120, b=120),
        updatemenus=[
            dict(
                buttons=metric_buttons,
                direction='down',
                showactive=True,
                x=0.01,
                y=1.22,
                xanchor='left',
                yanchor='top'
            )
        ],
        annotations=[
            dict(
                text='Quantiles: Q1 lowest, Q4 highest',
                xref='paper',
                yref='paper',
                x=1,
                y=1.22,
                showarrow=False,
                xanchor='right',
                yanchor='top',
                font=dict(size=11, color='white')
            )
        ]
    )
    correlation_fig.update_xaxes(title='', tickmode='array', tickvals=default_positions, ticktext=default_ticker_order, showticklabels=False, row=1, col=1)
    correlation_fig.update_xaxes(title='', tickmode='array', tickvals=default_positions, ticktext=default_ticker_order, tickangle=45, row=2, col=1)
    correlation_fig.update_yaxes(title='Correlation', row=1, col=1)
    correlation_fig.update_yaxes(title='Rank Shift', row=2, col=1)
    correlation_fig.show(config={'responsive': True})
else:
    print(f'No valid 10 year correlation data available for {benchmark}.')


In [ ]:
#8 Create DataFrames for each time frame
z_score_21 = pd.DataFrame()
z_score_50 = pd.DataFrame()
z_score_200 = pd.DataFrame()

# Calculate z-scores for 21-day time frame
z_score_sortino_ratio_21 = signal_labels.z_score(rolling_sortino_ratios_etf_21)
z_score_benchmark_minus_etf_21 = signal_labels.z_score(rolling_sortino_ratios_benchmark_minus_etf_21)

z_score_21['21 day Sortino Ratio (z score)'] = z_score_sortino_ratio_21
z_score_21['21 day Benchmark Minus ETF Sortino Ratio (z score)'] = z_score_benchmark_minus_etf_21
z_score_21.sort_values(by='21 day Sortino Ratio (z score)', ascending=True, inplace=True)

# Calculate z-scores for 50-day time frame
z_score_sortino_ratio_50 = signal_labels.z_score(rolling_sortino_ratios_etf_50)
z_score_benchmark_minus_etf_50 = signal_labels.z_score(rolling_sortino_ratios_benchmark_minus_etf_50)

z_score_50['50 day Sortino Ratio (z score)'] = z_score_sortino_ratio_50
z_score_50['50 day Benchmark Minus ETF Sortino Ratio (z score)'] = z_score_benchmark_minus_etf_50
z_score_50.sort_values(by='50 day Sortino Ratio (z score)', ascending=True, inplace=True)

# Calculate z-scores for 200-day time frame
correlation_column = f'10 year Correlation to {benchmark}'
z_score_200['200 day Sortino Ratio (z score)'] = signal_labels.z_score(rolling_sortino_ratios_etf_200)
z_score_200['200 day Benchmark Minus ETF Sortino Ratio (z score)'] = signal_labels.z_score(rolling_sortino_ratios_benchmark_minus_etf_200)
z_score_200[correlation_column] = correlation_10y
z_score_200.sort_values(by='200 day Sortino Ratio (z score)', ascending=True, inplace=True)

# Round decimal values
z_score_21 = z_score_21.round(2)
z_score_50 = z_score_50.round(2)
z_score_200 = z_score_200.round(2)

# Combine all time frames into one DataFrame
z_score_combined = pd.concat([z_score_21, z_score_50, z_score_200], axis=1)

# Sort columns by time frame and metric
#z_score_combined = z_score_combined.reindex(sorted(z_score_combined.columns, key=lambda x: (x.split()[0], x.split()[2])), axis=1)
#REVERSE ORDER OF COLUMNS (FROM RIGHT TO LEFT)
#CREATE FUNCTION THAT REINDEXES COLUMNS IN REVERSE ORDER
def reverse_column_order(df):
    df = df.reindex(columns=df.columns[::-1])
    return df

z_score_combined = reverse_column_order(z_score_combined)
if correlation_column in z_score_combined.columns:
    z_score_combined = z_score_combined[[
        column for column in z_score_combined.columns if column != correlation_column
    ] + [correlation_column]]

# Plot the combined metrics
combined_metrics_fig = qp.plot_z_score_combined(z_score_combined)
combined_metrics_fig.update_layout(autosize=True)
combined_metrics_fig.show(config={'responsive': True})
